<a href="https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M11/M11_Lab2_Beer_Game_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![Beer Game V2 banner](https://raw.githubusercontent.com/mdehghani86/DADS5250-GenAI/main/labs/M11/assets/images/M11_Lab2_Beer_Game_V2_banner.png)

# 🍺 Module 11 · Lab — Beer Game V2: Agentic Supply Chain

**Difficulty:** ⭐⭐⭐  ·  **Time:** ~15 min to run and analyze

In Module 4 you played the Beer Game with a **single LLM call per week**. Here you rebuild it with **memory-carrying agents** — one per role — and you get **two ordering strategies to compare**: a *Reactive* agent and a *Strategic* (base-stock + forecast) agent. Run them against the old one-shot rule, watch the round-by-round charts, and analyze which policy tames the **bullwhip effect**.


## 🔧 1. Setup

Install the course utils plus Anthropic (agent brains), Gradio (dashboard) and Plotly (charts). Run once per Colab runtime.


In [ ]:
# ==========================================================
# 1. Setup: install utils + agent/dashboard/plotting stack
# ==========================================================
%pip -q install dads5250==0.2.0 anthropic gradio plotly pandas numpy

import os, json, random                          # stdlib helpers
import numpy as np                               # demand distributions
import pandas as pd                              # weekly tables
import plotly.graph_objects as go                # interactive charts
from dads5250 import pp, pretty_print            # course utils


## ✅ 2. API check

Beer Game V2 uses **Claude** as the agent brain. The key is read from a Colab Secret named `ANTHROPIC_API_KEY`, then an environment variable, then a hidden prompt. On JupyterHub set `ANTHROPIC_API_KEY` via `export` or paste it at the prompt.


In [ ]:
# ==========================================================
# 2. API check: connect to Claude and show the model in use
# ==========================================================
import anthropic
from getpass import getpass

def _get_secret(name):
    try:                                          # 1) Colab Secret
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception:
        pass
    if os.environ.get(name):                      # 2) environment variable
        return os.environ[name]
    return getpass(f"Enter {name}: ")             # 3) hidden prompt

CLAUDE_MODEL = "claude-opus-4-8"                  # latest, most capable Claude
os.environ["ANTHROPIC_API_KEY"] = _get_secret("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()

try:
    _ = client.messages.create(model=CLAUDE_MODEL, max_tokens=8,
                               messages=[{"role":"user","content":"ping"}])
    status = "connected"
except Exception as e:
    status = f"NOT connected -- {type(e).__name__}"

pp({"Claude API": status, "model": CLAUDE_MODEL}, title="API check")


## 📋 3. The rules we keep from V1

To keep the comparison fair, **every physical rule is identical to the Module 4 Beer Game**. Only the *decision maker* changes.

| Rule | Value |
|------|-------|
| Roles (upstream order) | Retailer → Wholesaler → Distributor → Factory |
| Starting inventory | 10 / 15 / 20 / 25 |
| Holding cost | **1** per unit per week |
| Backorder cost | **2** per unit per week |
| Shipping lead time | 2 weeks |
| Demand | Market Fluctuation or Seasonal (verbatim from V1) |


In [ ]:
# ==========================================================
# 3. Rules carried over VERBATIM from the Module 4 Beer Game
# ----------------------------------------------------------
# Purpose: keep V2 physics identical to V1 so comparisons are fair.
# Defines:
#   - ROLES, INITIAL_INVENTORY, HOLDING_COST, BACKORDER_COST, LEAD_TIME
#   - define_demand_distribution() : V1's demand generator, unchanged
# ==========================================================
ROLES = ["Retailer", "Wholesaler", "Distributor", "Factory"]
INITIAL_INVENTORY = {"Retailer": 10, "Wholesaler": 15, "Distributor": 20, "Factory": 25}
HOLDING_COST   = 1     # per unit of inventory held, per week (V1)
BACKORDER_COST = 2     # per unit of unmet demand, per week (V1)
LEAD_TIME      = 2     # weeks for an order to arrive from upstream

def define_demand_distribution(demand_type, period=None):
    """Demand generator copied unchanged from the Module 4 Beer Game (V1)."""
    if demand_type == "Market Fluctuation":
        return int(np.random.randint(9, 15))                 # B=12, V=3
    elif demand_type == "Seasonal Demand":
        if period is None:
            raise ValueError("Period is required for Seasonal Demand.")
        cycle_phase = period % 5                              # 5-period cycle
        seasonal_factors = [
            np.random.randint(-4, 0), np.random.randint(-2, 1),
            np.random.randint(1, 2),  np.random.randint(3, 6),
            np.random.randint(-2, 2),
        ]
        return int(max(5, min(20, 11 + seasonal_factors[cycle_phase])))  # clamp 5..20
    raise ValueError("Invalid demand type selected.")


## 🧠 4. Two agent strategies to compare

A single LLM call per week (V1) has no memory, so small demand wobbles get amplified upstream — the **bullwhip effect**. V2 gives each role a memory-carrying agent, and lets you choose **how it reasons**:

- **⚡ Reactive** — myopic. Looks at the current inventory/backlog gap and the latest demand, and orders to close the gap. Simple, but tends to over-correct.
- **🎯 Strategic** — base-stock + forecast. Keeps a target stock level, **forecasts** next-period demand from its memory, and leans on the **downstream coordination signal** to smooth orders and damp the bullwhip.

Both strategies share the same memory and the same rules — the difference is the *policy*. That is exactly what you will analyze.


In [ ]:
# ==========================================================
# 4. BeerAgent: one memory-carrying agent per role, TWO strategies
# ----------------------------------------------------------
# Purpose: give students two decision policies to compare.
# Defines:
#   - BeerAgent(role, strategy)  strategy in {"reactive","strategic"}
#   - .observe()  : append this week's state to memory
#   - .decide()   : ask Claude for the order (falls back to a heuristic
#                   that MATCHES the chosen strategy if the API is down)
# ==========================================================
BASE_STOCK = {"Retailer": 20, "Wholesaler": 24, "Distributor": 28, "Factory": 32}

STRATEGY_BRIEF = {
 "reactive":  ("You are a REACTIVE ordering agent. Order only to close the gap between "
               "what you must serve now and what you have on hand. React to the latest "
               "demand; do not build for the long term."),
 "strategic": ("You are a STRATEGIC base-stock agent. Keep inventory near a target level, "
               "FORECAST next-period demand from your recent memory, and use the downstream "
               "coordination signal to SMOOTH orders and avoid over-reacting (fight the bullwhip)."),
}

class BeerAgent:
    def __init__(self, role, strategy, client=None, model=None):
        self.role, self.strategy = role, strategy
        self.client, self.model = client, model
        self.history = []                          # <-- persistent memory

    def observe(self, record):
        self.history.append(record)

    def _forecast(self):
        d = [h["incoming_order"] for h in self.history[-4:]]
        return sum(d)/len(d) if d else 10.0        # mean of recent demand

    def _trend(self):
        d = [h["incoming_order"] for h in self.history[-4:]]
        if len(d) < 2: return "unknown"
        return "rising" if d[-1] > d[0] else "falling" if d[-1] < d[0] else "stable"

    def _heuristic(self, inv, backlog, incoming, signal):
        """Deterministic policy per strategy -- also the API fallback."""
        if self.strategy == "reactive":
            adj = {"rising":1, "falling":-1, "stable":0, "unknown":0}[self._trend()]
            return max(0, incoming + backlog - inv + adj)   # chase demand, lean on trend
        target = BASE_STOCK[self.role]                       # base-stock policy
        forecast = self._forecast()
        gap = target - (inv - backlog)                       # how far below target
        raw = forecast + 0.5*gap + 0.25*(signal - forecast)  # smooth toward signal
        return max(0, int(round(raw)))

    def decide(self, inv, backlog, incoming, signal):
        if self.client is None:                              # offline mode -> heuristic
            return self._heuristic(inv, backlog, incoming, signal)
        prompt = f"""{STRATEGY_BRIEF[self.strategy]}
You are the {self.role} in a Beer Game supply chain.
Costs: holding={HOLDING_COST}/unit/week, backorder={BACKORDER_COST}/unit/week (backorder hurts 2x). Lead time {LEAD_TIME} weeks.
This week -> inventory {inv}, backlog {backlog}, order received from downstream {incoming}.
Demand trend {self._trend()}, forecast {self._forecast():.1f}, coordination signal {signal}.
Recent memory: {json.dumps(self.history[-6:])}
Order how many units from your upstream supplier this week to minimize cost. Reply with ONLY one non-negative integer."""
        try:
            m = self.client.messages.create(model=self.model, max_tokens=12,
                    messages=[{"role":"user","content":prompt}])
            txt = "".join(b.text for b in m.content if getattr(b,"type",None)=="text")
            return max(0, int("".join(c for c in txt if c.isdigit()) or 0))
        except Exception:
            return self._heuristic(inv, backlog, incoming, signal)


## 🔄 5. The simulation engine

The engine steps the chain week by week. Each week, per role: pipeline shipments arrive after the lead time, the role ships what it can against demand plus backlog, unmet demand becomes backlog, and cost accrues at the same 1/2 rates as V1. The only swap versus V1 is **who decides the order**.


In [ ]:
# ==========================================================
# 5. run_simulation: step the chain week by week
# ----------------------------------------------------------
# Defines:
#   - run_simulation(weeks, demand_type, agents) -> DataFrame
#     agents: dict role -> BeerAgent (or None to use a naive V1 rule)
#   - bullwhip_ratio(df) : order variability upstream vs customer demand
# ==========================================================
def run_simulation(weeks, demand_type, agents, seed=42):
    np.random.seed(seed); random.seed(seed)
    inv     = dict(INITIAL_INVENTORY)
    backlog = {r: 0 for r in ROLES}
    pipeline = {r: [0]*LEAD_TIME for r in ROLES}
    rows = []
    for wk in range(1, weeks+1):
        incoming_order = define_demand_distribution(demand_type, period=wk)  # customer demand
        signal = incoming_order                                              # shared signal down->up
        for role in ROLES:
            inv[role] += pipeline[role].pop(0)                              # arrival from LEAD_TIME ago
            need = incoming_order + backlog[role]
            shipped = min(inv[role], need); inv[role] -= shipped
            backlog[role] = need - shipped
            a = agents[role]
            if a is not None:
                a.observe({"inventory":inv[role],"backlog":backlog[role],"incoming_order":incoming_order})
                order = int(a.decide(inv[role], backlog[role], incoming_order, signal))
            else:
                order = max(0, incoming_order + backlog[role] - inv[role])   # naive V1 rule
            pipeline[role].append(order)
            cost = HOLDING_COST*max(inv[role],0) + BACKORDER_COST*backlog[role]
            rows.append({"week":wk,"role":role,"demand":incoming_order,
                         "inventory":inv[role],"backlog":backlog[role],"order":order,"cost":cost})
            incoming_order = order                                           # becomes next role's demand
    df = pd.DataFrame(rows)
    df["cum_cost"] = df.groupby("role")["cost"].cumsum()
    return df

def bullwhip_ratio(df):
    """Order variance at the Factory divided by customer-demand variance (>=1 = amplified)."""
    cust = df[df.role=="Retailer"]["demand"].var()
    fac  = df[df.role=="Factory"]["order"].var()
    return round((fac or 0)/(cust or 1), 2)


## 📊 6. The dashboard

Pick a **strategy** and a demand pattern, set the number of weeks, and hit **Run**. You get headline metrics (total cost, peak backlog, bullwhip ratio), four live charts, a **round-by-round orders** view, and a full decision log — so every round is easy to read.


In [ ]:
# ==========================================================
# 6. Gradio dashboard: strategy picker + live charts + round view
# ==========================================================
import gradio as gr
ROLE_ICON = {"Retailer":"🛒","Wholesaler":"🏬","Distributor":"🚚","Factory":"🏭"}
COLORS    = {"Retailer":"#2563eb","Wholesaler":"#059669","Distributor":"#d97706","Factory":"#7c3aed"}

def _line(df, col, title):
    fig = go.Figure()
    for r in ROLES:
        d = df[df.role==r]
        fig.add_trace(go.Scatter(x=d.week, y=d[col], mode="lines+markers",
                                 name=f"{ROLE_ICON[r]} {r}", line=dict(color=COLORS[r], width=3)))
    fig.update_layout(title=title, template="plotly_white", height=300,
                      margin=dict(l=40,r=20,t=45,b=30), legend=dict(orientation="h", y=-0.2))
    return fig

def _rounds(df):
    """Grouped bar: each role's order in every round (week)."""
    fig = go.Figure()
    for r in ROLES:
        d = df[df.role==r]
        fig.add_trace(go.Bar(x=d.week, y=d.order, name=f"{ROLE_ICON[r]} {r}", marker_color=COLORS[r]))
    fig.update_layout(title="🎲 Orders placed each round", barmode="group", template="plotly_white",
                      height=320, margin=dict(l=40,r=20,t=45,b=30), legend=dict(orientation="h", y=-0.2))
    return fig

def play(strategy_label, demand_type, weeks):
    strategy = "reactive" if strategy_label.startswith("⚡") else "strategic"
    agents = {r: BeerAgent(r, strategy, client, CLAUDE_MODEL) for r in ROLES}
    df = run_simulation(int(weeks), demand_type, agents)
    total, peak, bw = int(df.cost.sum()), int(df.backlog.max()), bullwhip_ratio(df)
    cards = (f"<div style='display:flex;gap:14px;flex-wrap:wrap'>"
             f"<div style='flex:1;min-width:150px;background:#eff6ff;border-radius:12px;padding:14px'>"
             f"<div style='font-size:12px;color:#64748b'>TOTAL COST</div>"
             f"<div style='font-size:26px;font-weight:700;color:#2563eb'>{total}</div></div>"
             f"<div style='flex:1;min-width:150px;background:#fef2f2;border-radius:12px;padding:14px'>"
             f"<div style='font-size:12px;color:#64748b'>PEAK BACKLOG</div>"
             f"<div style='font-size:26px;font-weight:700;color:#dc2626'>{peak}</div></div>"
             f"<div style='flex:1;min-width:150px;background:#f5f3ff;border-radius:12px;padding:14px'>"
             f"<div style='font-size:12px;color:#64748b'>BULLWHIP RATIO</div>"
             f"<div style='font-size:26px;font-weight:700;color:#7c3aed'>{bw}×</div></div></div>")
    log = df[["week","role","demand","inventory","backlog","order","cost"]]
    return (cards, _line(df,"inventory","📦 Inventory by week"),
            _line(df,"backlog","⛔ Backlog by week"),
            _rounds(df), _line(df,"cum_cost","💰 Cumulative cost by week"), log)

with gr.Blocks(theme=gr.themes.Soft(), title="Beer Game V2") as demo:
    gr.Markdown("# 🍺 Beer Game V2 — Agentic Supply Chain\nChoose a strategy, run the game, and read the rounds.")
    with gr.Row():
        strategy = gr.Radio(["⚡ Reactive","🎯 Strategic"], value="🎯 Strategic", label="Agent strategy")
        demand_type = gr.Dropdown(["Market Fluctuation","Seasonal Demand"],
                                  value="Market Fluctuation", label="Demand pattern")
        weeks = gr.Slider(6, 24, value=12, step=1, label="Weeks")
        run = gr.Button("▶ Run", variant="primary", scale=1)
    metrics = gr.HTML()
    with gr.Row():
        c1 = gr.Plot(); c2 = gr.Plot()
    with gr.Row():
        c3 = gr.Plot(); c4 = gr.Plot()
    log = gr.Dataframe(label="🧾 Decision log — every role, every round")
    run.click(play, [strategy, demand_type, weeks], [metrics, c1, c2, c3, c4, log])

demo.launch(debug=False)


## ⚖️ 7. Three-way comparison — the payoff

Run all three decision makers on the **same demand**: the naive V1 one-shot rule, the **Reactive** agent, and the **Strategic** agent. Lower total cost and a bullwhip ratio closer to **1×** are better.


In [ ]:
# ==========================================================
# 7. V1 vs Reactive vs Strategic on identical demand
# ==========================================================
def run_named(kind):
    if kind == "V1 (one-shot)":
        agents = {r: None for r in ROLES}
    else:
        strat = "reactive" if kind.startswith("Reactive") else "strategic"
        agents = {r: BeerAgent(r, strat, client, CLAUDE_MODEL) for r in ROLES}
    return run_simulation(12, "Market Fluctuation", agents)

results = {k: run_named(k) for k in ["V1 (one-shot)","Reactive agent","Strategic agent"]}
summary = {k: {"total_cost": int(v.cost.sum()), "bullwhip": bullwhip_ratio(v)} for k,v in results.items()}
pp(summary, title="V1 vs Reactive vs Strategic (same demand)")

fig = go.Figure(go.Bar(x=list(summary), y=[s["total_cost"] for s in summary.values()],
                       marker_color=["#94a3b8","#d97706","#7c3aed"],
                       text=[s["total_cost"] for s in summary.values()], textposition="outside"))
fig.update_layout(title="💰 Total supply-chain cost by decision maker (lower is better)",
                  template="plotly_white", height=360, yaxis_title="Total cost")
fig.show()


## 🎯 8. Exercises

1. **Observe:** Run each strategy on *Seasonal Demand* for 18 weeks. Which strategy keeps the bullwhip ratio lowest, and where does the backlog pile up?
2. **Code:** Tune the `BASE_STOCK` targets for the Strategic agent. Can you get total cost below the Reactive agent on both demand patterns?
3. **Analyze:** Find a demand pattern or week count where **Reactive beats Strategic**. Explain why the simpler policy wins there.

## 📝 Summary

You rebuilt the Beer Game as an **agentic system** with two comparable strategies — Reactive and Strategic — memory, forecasting, and coordination, all wrapped in an easy Gradio dashboard with round-by-round charts and full tracking. Same rules as Module 4; the analysis is now about **which policy wins, and why**.
